In [ ]:
# Download IndicSynth Dataset
import sys
import types
for k in list(sys.modules.keys()):
    if k.startswith("datasets") or k.startswith("fsspec") or k == "lzma" or k == "_lzma":
        del sys.modules[k]

fake_lzma = types.ModuleType("lzma")
fake_lzma.LZMAError = Exception
class FakeLZMAFile:
    def __init__(self, *args, **kwargs): pass
    def read(self, *args, **kwargs): return b""
    def write(self, *args, **kwargs): return 0
    def seek(self, *args, **kwargs): return 0
    def tell(self, *args, **kwargs): return 0
    def close(self): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
fake_lzma.LZMAFile = FakeLZMAFile
fake_lzma.open = lambda *args, **kwargs: FakeLZMAFile()
fake_lzma.is_check_supported = lambda check: False
fake_lzma.FORMAT_AUTO, fake_lzma.FORMAT_XZ, fake_lzma.FORMAT_ALONE, fake_lzma.FORMAT_RAW = 0, 1, 2, 3
fake_lzma.CHECK_NONE, fake_lzma.CHECK_CRC32, fake_lzma.CHECK_CRC64, fake_lzma.CHECK_SHA256 = 0, 1, 4, 10
sys.modules["lzma"] = fake_lzma
sys.modules["_lzma"] = fake_lzma
sys.modules["torchvision"] = None

# 1. INDICSYNTH PIPELINE
import os
import shutil
import random
from collections import defaultdict
from dotenv import load_dotenv
from datasets import load_dataset, Audio, ClassLabel

BASE_DIR = r"E:\SIH2026\models"
CACHE_DIR = os.path.join(BASE_DIR, "temp_cache")

os.environ["HF_HOME"] = CACHE_DIR
os.environ["HF_DATASETS_CACHE"] = CACHE_DIR
os.environ["HF_HUB_CACHE"] = CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_DIR
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.makedirs(CACHE_DIR, exist_ok=True)

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

LANGUAGE = "bengali"  # Change to hindi, telugu, etc.
SAMPLES_PER_CLASS = 4000  

# FIX: Set output directory format properly
OUTPUT_DIR = os.path.join(BASE_DIR, f"{LANGUAGE}_indicsynth")

print(f"Phase 1: Streaming {LANGUAGE.capitalize()} IndicSynth dataset...")

ds = load_dataset(
    "vdivyasharma/IndicSynth",
    name=LANGUAGE.capitalize(),
    split="train[:25%]+train[-25%:]",
    token=HF_TOKEN,
    cache_dir=CACHE_DIR
)
ds = ds.cast_column("audio", Audio(decode=False))

def assign_label(batch):
    labels = []
    for model in batch["Generative Model"]:
        if model is None or model == "" or model == "null":
            labels.append(0) # Bonafide
        else:
            labels.append(1) # Spoof
    batch["label"] = labels
    return batch

ds = ds.map(assign_label, batched=True, batch_size=1000, desc="Assigning Labels")
ds = ds.cast_column("label", ClassLabel(names=["bonafide", "spoof"]))

speaker_indices = defaultdict(lambda: {"bonafide": [], "spoof": []})
total_bonafide, total_spoof = 0, 0

for idx, item in enumerate(ds):
    spk_id = item.get("Target Speaker ID") or item.get("Source Speaker_ID")
    if not spk_id: continue  
    label = item["label"]
    
    if label == 0 and total_bonafide < SAMPLES_PER_CLASS:
        speaker_indices[spk_id]["bonafide"].append(idx)
        total_bonafide += 1
    elif label == 1 and total_spoof < SAMPLES_PER_CLASS:
        speaker_indices[spk_id]["spoof"].append(idx)
        total_spoof += 1
            
    if total_bonafide >= SAMPLES_PER_CLASS and total_spoof >= SAMPLES_PER_CLASS: break

all_speakers = list(speaker_indices.keys())
random.seed(42)
random.shuffle(all_speakers)

split_idx = int(0.8 * len(all_speakers))
train_speakers = set(all_speakers[:split_idx])
val_speakers = set(all_speakers[split_idx:])

train_idx_list, val_idx_list = [], []
for spk_id in train_speakers:
    train_idx_list.extend(speaker_indices[spk_id]["bonafide"] + speaker_indices[spk_id]["spoof"])
for spk_id in val_speakers:
    val_idx_list.extend(speaker_indices[spk_id]["bonafide"] + speaker_indices[spk_id]["spoof"])

random.shuffle(train_idx_list)
random.shuffle(val_idx_list)

train_dataset = ds.select(train_idx_list).cast_column("audio", Audio(sampling_rate=16000, decode=True))
val_dataset = ds.select(val_idx_list).cast_column("audio", Audio(sampling_rate=16000, decode=True))

os.makedirs(OUTPUT_DIR, exist_ok=True)
train_dataset.save_to_disk(os.path.join(OUTPUT_DIR, "train"))
val_dataset.save_to_disk(os.path.join(OUTPUT_DIR, "validation"))

print(f"\n{LANGUAGE.capitalize()} dataset successfully written to {OUTPUT_DIR}!")
if os.path.exists(CACHE_DIR): shutil.rmtree(CACHE_DIR)

In [ ]:
# Download Indic TTS Dataset
import os
import random
import soundfile as sf
import numpy as np
from datasets import load_dataset
from dotenv import load_dotenv

E_CACHE_DIR = r"E:\SIH2026\models\temp_cache"
E_TMP_DIR = r"E:\SIH2026\models\temp_files"
os.makedirs(E_CACHE_DIR, exist_ok=True)
os.makedirs(E_TMP_DIR, exist_ok=True)
os.environ["HF_HOME"] = E_CACHE_DIR

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

BASE_DIR = r"E:\SIH2026\models"
REPO_NAME = "SherryT997/IndicTTS-Deepfake-Challenge-Data"

# FIX: Proper output folders matching training paths
TARGET_LANGS = {
    "Hindi": "hindi_indicTTS",
    "Bengali": "bengali_indicTTS",
    "Telugu": "telugu_indicTTS"
}

MAX_PER_CLASS = 800
counts = {lang: {"real": 0, "fake": 0} for lang in TARGET_LANGS.values()}

print(f"Connecting to stream for {REPO_NAME}...")
stream_ds = load_dataset(
    REPO_NAME,
    split="train",
    streaming=True,
    token=HF_TOKEN,
    cache_dir=E_CACHE_DIR
)

def get_split():
    r = random.random()
    if r < 0.80: return "train"
    elif r < 0.90: return "val"
    else: return "test"

print("Streaming in progress...")
total_scanned = 0

for item in stream_ds:
    total_scanned += 1
    raw_lang = item.get("language", "")
    
    if raw_lang not in TARGET_LANGS: continue

    folder_name = TARGET_LANGS[raw_lang]
    is_tts = int(item["is_tts"])
    subfolder = "fake" if is_tts == 1 else "real"

    if counts[folder_name][subfolder] >= MAX_PER_CLASS: continue

    split = get_split()
    target_dir = os.path.join(BASE_DIR, folder_name, split, subfolder)
    os.makedirs(target_dir, exist_ok=True)

    audio_array = np.array(item["audio"]["array"], dtype=np.float32)
    sr = item["audio"]["sampling_rate"]

    counts[folder_name][subfolder] += 1
    file_idx = counts[folder_name][subfolder]
    file_path = os.path.join(target_dir, f"{split}_{file_idx:05d}.wav")

    sf.write(file_path, audio_array, sr)
    
    all_done = all(counts[l]["real"] >= MAX_PER_CLASS and counts[l]["fake"] >= MAX_PER_CLASS for l in TARGET_LANGS.values())
    if all_done: break

print("\nIndic audio files downloaded and partitioned into train/val/test folders.")

In [1]:
""" For English language Dataset visit the following link by Kaggle : https://www.kaggle.com/datasets/abdallamohamed312/in-the-wild-dataset """

' For English language Dataset visit the following link by Kaggle : https://www.kaggle.com/datasets/abdallamohamed312/in-the-wild-dataset '